# Retrieval-Augmented Generation (RAG)

<sup>2nd part of the DGT Summer School 2026 material.</sup>

---

Large Language Models are remarkable at reasoning, but they have one fundamental limitation: **their knowledge is frozen at the time of training**. Once trained, the model cannot learn new facts unless it is retrained or fine-tuned — both expensive operations.

**Retrieval-Augmented Generation (RAG)** solves this by giving the model access to an external knowledge base *at inference time*. Instead of relying solely on memorised parameters, the model can look up relevant passages and ground its answer in real documents.

> **Analogy.** RAG is to an LLM what an **open-book exam** is to a student. The student (model) is not expected to memorise every fact; instead, they are allowed to consult textbooks (knowledge base) and reason from what they find.
> 
> **Another analogy.** RAG is to an LLM what a **translation memory** is to a translator. The translator (model) is not expected to memorise every previous translation; instead, they are allowed to consult similar ones from before (knowledge base) and reuse the translations, style etc. from what they find.

---

## What you will learn

| Section | Topic |
|---|---|
| 1 | The hallucination problem: why pure LLMs fail on factual questions |
| 2 | Loading and inspecting a knowledge base (Wikipedia) |
| 3 | Text splitting strategies and why chunk size matters |
| 4 | Retrieval with **BM25** — a lexical retriever that runs locally |
| 5 | Building a RAG chain with modern **LCEL** syntax |
| 6 | Without RAG vs. with RAG — a direct comparison |
| 7 | Conversational RAG with memory |
| 8 | Inspecting what gets retrieved |
| 9 | Chunking strategy comparison |
| 10 | Summary |

> **Note on the retriever.** This notebook uses **BM25**, a classic *lexical* (keyword) retriever that needs **no embedding model and no extra API** — it runs entirely locally from the text. Only the chat model calls an external API. (In production you might instead — or additionally — use a *dense* embedding-based retriever; we mention the trade-off in the summary.)

---

## RAG Architecture

The RAG pipeline has two phases:

**Indexing (done once, offline)**
```
Documents  →  Text Splitter  →  BM25 index
```

**Retrieval + Generation (done at query time)**
```
User Query  →  BM25 Retriever  →  Top-k Chunks
                                        ↓
                  Prompt = [System + Context + Query]  →  LLM  →  Answer
```

The key insight is that the retriever finds the most relevant chunks for a query. **BM25** scores each chunk by weighted term overlap with the query: rare shared terms count for more, and the score is normalised for chunk length. It is cheap, transparent, and a strong baseline whenever the answer shares vocabulary with the question.

## 1  Setup

We only need a handful of lightweight libraries. The **retriever** (BM25) runs locally with no model download and no API. The **chat model** is the only component that calls an external API.

| Library | Purpose |
|---|---|
| `langchain` / `langchain-community` / `langchain-core` | Document loaders, chains, retrievers |
| `langchain-openai` | LangChain <-> OpenAI bridge (chat model) |
| `langchain-text-splitters` | Splitting documents into chunks |
| `rank-bm25` | BM25 lexical retrieval (the retriever we use) |
| `wikipedia` | Wikipedia document loader |
| `tiktoken` | Token counting (used by the token splitter) |
| `python-dotenv` | Loads your `OPENAI_API_KEY` from a local `.env` file |

In [1]:
# !pip install -q -U langchain langchain-community langchain-core langchain-text-splitters langchain-openai
# !pip install -q -U rank-bm25 wikipedia tiktoken python-dotenv

In [1]:
# Load your OpenAI API key from a local .env file (which must NOT be committed).
# The .env file should contain a single line:  OPENAI_API_KEY=sk-...
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - add it to your .env file."
print("OpenAI API key loaded.")

OpenAI API key loaded.


In [2]:
import warnings
warnings.filterwarnings("ignore")

from langchain_openai import ChatOpenAI
from langchain_community.retrievers import BM25Retriever
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

print("Imports ready.")

Imports ready.


## 2  The Problem: LLMs Hallucinate Facts

Before adding any retrieval, let's demonstrate the core problem that RAG solves.

LLMs generate text by predicting the next token based on patterns in training data. When asked a factual question about something they don't know well, they don't say "I don't know" - they **hallucinate** plausible-sounding but wrong answers.

We will use OpenAI's **`gpt-5.1`** as our LLM. It is called over the API, so nothing is downloaded or run locally - you only need your `OPENAI_API_KEY`.

We set `temperature=0` so answers are (nearly) deterministic: the same prompt yields the same answer, which keeps the *without-RAG* vs. *with-RAG* comparison fair.

In [3]:
# Model id and endpoint are read from .env (OPENAI_MODEL, OPENAI_BASE_URL),
# so the same code works against OpenAI or the course's custom endpoint.
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.1")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = ChatOpenAI(model=LLM_MODEL, base_url=BASE_URL, temperature=0, max_tokens=1024)

print(f"Chat model ready: {LLM_MODEL}  (endpoint: {BASE_URL})")

Chat model ready: gpt-4o-mini  (endpoint: https://api.openai.com/v1)


In [4]:
# Quick sanity check that the API key and model work.
print(llm.invoke("Reply with exactly: RAG notebook is ready.").content)

RAG notebook is ready.


In [5]:
# A plain (no-retrieval) chain: prompt -> LLM -> string.
# ChatOpenAI takes structured chat messages, so we build them with ChatPromptTemplate
# rather than a model-specific chat template.

plain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question concisely and accurately."),
    ("human", "{question}"),
])

plain_chain = plain_prompt | llm | StrOutputParser()

In [7]:
# Ask a highly specific factual question — something the model may not know well
question = "What are the key differences between BERT and RoBERTa in terms of training procedure?"

print("=== WITHOUT RAG ===")
answer_no_rag = plain_chain.invoke({"question": question})
print(answer_no_rag)

=== WITHOUT RAG ===


BERT and RoBERTa are both transformer-based models for natural language processing, but they differ in their training procedures in several key ways:

1. **Training Data**:
   - **BERT**: Trained on the BooksCorpus and English Wikipedia.
   - **RoBERTa**: Trained on a larger dataset that includes the same sources as BERT but also incorporates additional data from the Common Crawl, resulting in a more extensive and diverse training corpus.

2. **Training Objective**:
   - **BERT**: Uses the masked language model (MLM) objective and next sentence prediction (NSP) during training.
   - **RoBERTa**: Uses only the masked language model (MLM) objective and removes the next sentence prediction task, which has been shown to be less beneficial.

3. **Training Duration and Batch Size**:
   - **BERT**: Trained with a fixed number of training steps and a specific batch size.
   - **RoBERTa**: Trained for a longer duration with larger batch sizes, which helps improve performance.

4. **Dynamic Mask

In [8]:
# Ask a highly specific factual question — something the model may not know well
question = "In what year was the founding stallion Siglavy foaled and where did he originally come from?"

print("=== WITHOUT RAG ===")
answer_no_rag = plain_chain.invoke({"question": question})
print(answer_no_rag)

=== WITHOUT RAG ===


The founding stallion Siglavy was foaled in 1878 and originally came from the Arabian Peninsula, specifically from the region of Syria.


The model may give a reasonable-sounding answer, but it is reconstructing facts from memory — which can contain errors or omissions. Now let's build a RAG system that **retrieves the actual source text** before answering.

## 3  Building the Knowledge Base

### 3.1  Loading Documents

We use **Wikipedia** as our knowledge source. It has two advantages for demos:

1. No authentication or scraping needed — `WikipediaLoader` fetches articles via the Wikipedia API.
2. The content is well-structured and factual, making it easy to verify whether the model's answer is grounded.

We load several NLP-related articles to give our RAG system a rich knowledge base.

In [9]:
# Wikipedia's API now requires a descriptive User-Agent header, otherwise it
# returns HTTP 403 (see https://w.wiki/4wJS). Set one before loading.
import wikipedia
wikipedia.set_user_agent("DGT-Summer-School/1.0 (NLP course material; contact: ales.zagar@fri.uni-lj.si)")

# Articles to include in our knowledge base
topics = [
    "BERT (language model)",
    "GPT (language model)",
    "Transformer (deep learning architecture)",
    "Word2vec",
    "Attention mechanism",
    "Transfer learning",
    "Natural language processing",
    "Lipizzan",
    "Primož Trubar",
    "Idrija mercury mine",
]

print("Fetching Wikipedia articles...")
all_docs = []
for topic in topics:
    loader = WikipediaLoader(query=topic, load_max_docs=1, doc_content_chars_max=30000)
    docs = loader.load()
    all_docs.extend(docs)
    print(f"  Loaded: '{topic}' ({len(docs[0].page_content)} chars)")

print(f"\nTotal documents: {len(all_docs)}")
print(f"Total characters: {sum(len(d.page_content) for d in all_docs):,}")

Fetching Wikipedia articles...


  Loaded: 'BERT (language model)' (17537 chars)


  Loaded: 'GPT (language model)' (14543 chars)


  Loaded: 'Transformer (deep learning architecture)' (30000 chars)


  Loaded: 'Word2vec' (29636 chars)


...

  Loaded: 'Transfer learning' (8654 chars)


  Loaded: 'Natural language processing' (30000 chars)


  Loaded: 'Lipizzan' (22504 chars)


  Loaded: 'Primož Trubar' (7430 chars)


  Loaded: 'Idrija mercury mine' (6941 chars)

Total documents: 10
Total characters: 192,836


In [10]:
# Inspect one document
print("=== Document metadata ===")
print(all_docs[0].metadata)
print("\n=== First 500 characters ===")
print(all_docs[0].page_content[:500])

=== Document metadata ===
{'title': 'BERT (language model)', 'summary': 'Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved the state of the art for large language models. As of 2020, BERT is a ubiquitous baseline in natural language processing (NLP) experiments. \nBERT is trained by masked token prediction and next sentence prediction. With this training, BERT learns contextual, latent representations of tokens in their context, similar to ELMo and GPT-2. It found applications for many natural language processing tasks, such as coreference resolution and polysemy resolution. It improved on ELMo and spawned the study of "BERTology", which attempts to interpret what is learned by BERT.\nBERT was originally implemented in the English language a

### 3.2  Text Splitting

The LLM context window has a **token limit**, and long chunks dilute the signal we hand the model. A full Wikipedia article may have 10,000+ characters — too long to feed to the retriever or the prompt as a single unit.

We must split documents into **chunks**: shorter, self-contained passages that can be:
1. Scored individually by the retriever.
2. Passed selectively to the LLM based on relevance.

**Key parameters:**

| Parameter | Effect |
|---|---|
| `chunk_size` | Maximum size of each chunk (in characters or tokens) |
| `chunk_overlap` | How many characters from the end of one chunk are repeated at the start of the next |

**Why overlap?** Important context may straddle a chunk boundary. Overlap ensures that such information appears in at least one complete chunk. Too much overlap wastes space; too little risks missing cross-boundary context.

**Choosing chunk size:**
- **Too small** → chunks may lack enough context to answer a question; retrieval may be noisy.
- **Too large** → fewer, denser chunks; important passages are diluted by surrounding text; may overflow the context window.
- A common starting point: **512–1024 characters** (roughly 100–200 tokens).

In [11]:
# RecursiveCharacterTextSplitter tries to split on natural boundaries first:
# paragraphs (\n\n) → sentences (\n) → words ( ) → characters
# This preserves semantic coherence better than a hard character cut.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,          # Target size in characters
    chunk_overlap=100,       # 100-char overlap between consecutive chunks
    length_function=len,
    add_start_index=True,    # Store the original char offset in metadata
)

chunks = text_splitter.split_documents(all_docs)

print(f"Documents → chunks: {len(all_docs)} → {len(chunks)}")
print(f"Average chunk size : {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
print(f"Min / Max          : {min(len(c.page_content) for c in chunks)} / {max(len(c.page_content) for c in chunks)} chars")

Documents → chunks: 10 → 380
Average chunk size : 519 chars
Min / Max          : 5 / 799 chars


### 3.3  Comparing Chunk Sizes

To build intuition, let's see how different `chunk_size` values affect the number and character of the chunks.

In [12]:
print(f"{'chunk_size':>12}  {'num_chunks':>10}  {'avg_len':>8}")
print("-" * 36)
for size in [200, 400, 800, 1600]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 8)
    ch = sp.split_documents(all_docs)
    avg = sum(len(c.page_content) for c in ch) / len(ch)
    print(f"{size:>12}  {len(ch):>10}  {avg:>8.0f}")

  chunk_size  num_chunks   avg_len
------------------------------------
         200        1353       146
         400         739       271
         800         380       519


        1600         180      1092

## 4  Retrieval with BM25

The retriever takes a query and returns the most relevant chunks from our knowledge base. We use **BM25**, a classic *lexical* (keyword) ranking function. It needs **no embedding model, no GPU and no external API** — it builds directly from the tokenised chunks and runs entirely locally.

**How BM25 scores a chunk** against the query, in two intuitions:

- **Rare terms count more** (inverse document frequency): matching "Siglavy" is far more informative than matching "the".
- **Term frequency saturates** and is **length-normalised**, so a long chunk doesn't win just by repeating a query word or by being long.

BM25 returns the top-k chunks by score. It is a strong baseline and the natural choice whenever the answer tends to share vocabulary with the question — exactly the case for the factual questions in this notebook. Let's build one over our `chunks`.

In [13]:
# Build a BM25 retriever directly from the chunks — no embedding model, no API.
# BM25 scores each chunk by weighted keyword overlap with the query.
import re

def bm25_preprocess(text):
    # Tokenise for BM25: lowercase + split on word characters. This matters —
    # the default tokeniser is a plain str.split, so "Siglavy:" (with a colon)
    # or "BERT" (uppercase) would NOT match a "siglavy"/"bert" query term.
    return re.findall(r"\w+", text.lower())

bm25_retriever = BM25Retriever.from_documents(chunks, preprocess_func=bm25_preprocess)
bm25_retriever.k = 4   # number of chunks to return per query

query = "How does BERT differ from GPT in terms of training objectives?"
bm25_results = bm25_retriever.invoke(query)

print("=== BM25 Search Results ===")
for i, doc in enumerate(bm25_results):
    title = doc.metadata.get("title", "unknown")
    print(f"[{i+1}] Source: {title}")
    print(doc.page_content[:200])
    print()

=== BM25 Search Results ===
[1] Source: Generative pre-trained transformer
own products with "GPT", but it has begun enabling its ChatGPT Plus subscribers to make "custom versions of ChatGPT" called GPTs on the OpenAI site. OpenAI's terms of service says that its subscribers

[2] Source: Generative pre-trained transformer
OpenAI, which created the first generative pre-trained transformer (GPT) in 2018, asserted in 2023 that "GPT" should be regarded as a brand of OpenAI. In April 2023, OpenAI revised the brand guideline

[3] Source: BERT (language model)
BERT is trained by masked token prediction and next sentence prediction. With this training, BERT learns contextual, latent representations of tokens in their context, similar to ELMo and GPT-2. It fo

[4] Source: BERT (language model)
replaced with a [MASK] token with probability 80%,
replaced with a random word token with probability 10%,
not replaced with probability 10%.
The reason not all selected tokens are masked is to avoid 



## 5  Building the RAG Chain with LCEL

**LangChain Expression Language (LCEL)** is LangChain's modern way of composing chains using the pipe operator `|`. Each component receives output from the previous step:

```python
chain = step_1 | step_2 | step_3
chain.invoke(input)  # passes input through all steps
```

This replaces the older `ConversationalRetrievalChain` and similar high-level abstractions, which are now deprecated. LCEL is more transparent, composable, and supports streaming out of the box.

### Full RAG chain

```
Question
   ├──────────────────────────────────► RunnablePassthrough (keeps question)
   │                                             │
   └──► Retriever ──► format_docs (join)         │
                           │                     │
                    {context: ..., question: ...} │
                                    ↓
                               PromptTemplate
                                    ↓
                                   LLM
                                    ↓
                              StrOutputParser
```

In [14]:
# RAG prompt as a ChatPromptTemplate. It receives a dict {context, question}.

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     """You are a precise, helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information to answer that."
Do not make up facts. Cite which source(s) you used at the end of your answer.

CONTEXT:
{context}"""),
    ("human", "{question}"),
])

In [15]:
# Our retriever is a BM25 index over the chunks (see §4). Ask it for the top-5.
retriever = BM25Retriever.from_documents(chunks, preprocess_func=bm25_preprocess)
retriever.k = 5

# Helper to concatenate retrieved Documents into a single context string
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('title', 'unknown')}]\n{d.page_content}"
        for d in docs
    )

In [16]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

In [17]:
# Helper to print the answer neatly
def ask(question: str):
    print(f"Question: {question}")
    print("-" * 60)
    answer = rag_chain.invoke(question)
    print(answer)
    print("=" * 60)
    return answer

In [18]:
ask("What are the key differences between BERT and GPT in terms of their architecture and training objectives?")

Question: What are the key differences between BERT and GPT in terms of their architecture and training objectives?
------------------------------------------------------------


I don't have enough information to answer that.


"I don't have enough information to answer that."

In [19]:
ask("What is the attention mechanism and why was it introduced?")

Question: What is the attention mechanism and why was it introduced?
------------------------------------------------------------


The attention mechanism is a technique used in neural networks, particularly in sequence-to-sequence models, to allow the model to focus on different parts of the input sequence when producing an output. It was introduced to solve the bottleneck problem of fixed-size output vectors in recurrent neural networks (RNNs), enabling the model to process long-distance dependencies more easily. This mechanism allows the model to "emulate searching through a source sentence" during the decoding of a translation, improving the quality of the output.

The attention mechanism was introduced in the context of machine translation to enhance the performance of models by allowing them to consider the entire input sequence rather than being constrained by a fixed-size representation. 

Sources: [Source: Transformer (deep learning)], [Source: Attention (machine learning)]


'The attention mechanism is a technique used in neural networks, particularly in sequence-to-sequence models, to allow the model to focus on different parts of the input sequence when producing an output. It was introduced to solve the bottleneck problem of fixed-size output vectors in recurrent neural networks (RNNs), enabling the model to process long-distance dependencies more easily. This mechanism allows the model to "emulate searching through a source sentence" during the decoding of a translation, improving the quality of the output.\n\nThe attention mechanism was introduced in the context of machine translation to enhance the performance of models by allowing them to consider the entire input sequence rather than being constrained by a fixed-size representation. \n\nSources: [Source: Transformer (deep learning)], [Source: Attention (machine learning)]'

In [20]:
ask("How does Word2vec represent words as vectors?")

Question: How does Word2vec represent words as vectors?
------------------------------------------------------------


Word2vec represents a word as a high-dimension vector of numbers which capture relationships between words. Words that appear in similar contexts are mapped to vectors that are nearby as measured by cosine similarity, indicating the level of semantic similarity between the words. For example, the vectors for "walk" and "ran" are nearby, as are those for "but" and "however", and "Berlin" and "Germany" (Source: Word2vec).


'Word2vec represents a word as a high-dimension vector of numbers which capture relationships between words. Words that appear in similar contexts are mapped to vectors that are nearby as measured by cosine similarity, indicating the level of semantic similarity between the words. For example, the vectors for "walk" and "ran" are nearby, as are those for "but" and "however", and "Berlin" and "Germany" (Source: Word2vec).'

In [21]:
ask("In what year was the founding stallion Siglavy foaled and where did he originally come from?")

Question: In what year was the founding stallion Siglavy foaled and where did he originally come from?
------------------------------------------------------------


Siglavy was foaled in 1810 and originally came from Syria. [Source: Lipizzan]


'Siglavy was foaled in 1810 and originally came from Syria. [Source: Lipizzan]'

## 6  Without RAG vs. With RAG — Side-by-Side

This is the core experiment. We ask the same specific factual question to:
1. The plain LLM (no context)
2. The RAG-augmented LLM (with retrieved Wikipedia context)

After running both cells, compare:
- **Specificity**: Does the RAG answer include concrete details absent from the baseline?
- **Grounding**: Does the RAG answer cite sources and stick to them?
- **Accuracy**: Is either answer factually wrong?

In [22]:
comparison_question = "In what year was the founding stallion Siglavy foaled and where did he originally come from?"

print("=" * 60)
print("WITHOUT RAG (pure LLM memory)")
print("=" * 60)
baseline = plain_chain.invoke({"question": comparison_question})
print(baseline)

WITHOUT RAG (pure LLM memory)


The founding stallion Siglavy was foaled in 1878 and originally came from the Arabian Peninsula, specifically from the region of Syria.


In [23]:
print("=" * 60)
print("WITH RAG (retrieved Wikipedia context)")
print("=" * 60)
rag_answer = rag_chain.invoke(comparison_question)
print(rag_answer)

WITH RAG (retrieved Wikipedia context)


Siglavy was foaled in 1810 and originally came from Syria. [Source: Lipizzan]


In [24]:
# Also show what context was actually retrieved
retrieved = retriever.invoke(comparison_question)
print("=" * 60)
print(f"Retrieved {len(retrieved)} chunks:")
print("=" * 60)
for i, doc in enumerate(retrieved):
    print(f"\n[Chunk {i+1}] Source: {doc.metadata.get('title', 'unknown')}")
    print(doc.page_content[:1000], "...")

Retrieved 5 chunks:

[Chunk 1] Source: Lipizzan
Pluto: a gray Spanish stallion from the Royal Danish Stud, foaled in 1765
Conversano: a black Neapolitan stallion, foaled in 1767
Maestoso: a gray stallion from the Kladrub stud with a Spanish dam, foaled 1773, descendants today all trace via Maestoso X, foaled in Hungary in 1819
Favory: a dun stallion from the Kladrub stud, foaled in 1779
Neapolitano: a bay Neapolitan stallion from the Polesine, foaled in 1790
Siglavy: a gray Arabian stallion, originally from Syria, foaled in 1810
Two additional stallion lines are found in Croatia, Hungary, and other eastern European countries, as well as in North America. They are accepted as equal to the six classical lines by the Lipizzan International Federation. These are: ...

[Chunk 2] Source: Primož Trubar
Trubar was born in the village of Rašica (now in the Municipality of Velike Lašče) in the Duchy of Carniola, then under the Habsburgs. In the years 1520–1521 he attended school in Rijeka, in 15

## 7  Conversational RAG

So far our RAG chain handles single-turn questions. But real assistants need to maintain conversation history — a follow-up question like "Tell me more about that" only makes sense if the model remembers what "that" referred to.

**The challenge:** If the user says "How was it trained?" in a follow-up, the retriever receives this ambiguous query and may retrieve irrelevant chunks. We need to **reformulate** the question using the chat history before retrieval.

The modern LangChain approach uses two chains:
1. **History-aware retriever**: rewrites the follow-up question into a standalone question using chat history.
2. **Question-answering chain**: answers the standalone question using retrieved context.

```
Chat History + Follow-up Question
          ↓
  Reformulation Prompt + LLM
          ↓
   Standalone Question
          ↓
      Retriever
          ↓
  RAG Answer Chain
```

In [25]:
# Step 1: Rewrite a follow-up question into a standalone question using chat history.

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the conversation so far and a follow-up question, reformulate the follow-up "
     "as a standalone question that can be understood without the history. "
     "Return ONLY the reformulated question, nothing else."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

contextualize_chain = contextualize_prompt | llm | StrOutputParser()

def contextualized_question(input_dict: dict) -> str:
    """If there is chat history, rewrite the question; otherwise pass it through."""
    if input_dict.get("chat_history"):
        return contextualize_chain.invoke(input_dict)
    return input_dict["input"]

In [26]:
# QA prompt for conversational RAG.

conv_rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a helpful NLP tutor. Answer the question using only the context below.
If the answer is not in the context, say you don't know. Be concise.

CONTEXT:
{context}"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

conversational_rag_chain = (
    RunnablePassthrough.assign(
        context=RunnableLambda(contextualized_question) | retriever | format_docs
    )
    | conv_rag_prompt
    | llm
    | StrOutputParser()
)

In [27]:
# Stateful conversation loop
chat_history = []

def chat_with_rag(user_message: str) -> str:
    # The chain now returns a string directly (pure LCEL, no wrapper dict)
    answer = conversational_rag_chain.invoke({
        "input": user_message,
        "chat_history": chat_history,
    })
    # Append the exchange to history for next turn
    chat_history.append(HumanMessage(content=user_message))
    chat_history.append(AIMessage(content=answer))
    return answer

In [28]:
# Turn 1: Initial question
q1 = "What is the Transformer architecture?"
print(f"User: {q1}")
a1 = chat_with_rag(q1)
print(f"Assistant: {a1}")

User: What is the Transformer architecture?


Assistant: The Transformer architecture is a deep learning model developed for natural language processing (NLP). It was introduced in the paper "Attention Is All You Need" in 2017 and solved many performance issues associated with older recurrent neural network (RNN) designs. The architecture uses an attention mechanism that allows models to process entire sequences of text at once, enabling the training of larger and more sophisticated models. It serves as the core technology for generative pre-trained transformers (GPT) and is widely used in various NLP tasks.


In [29]:
# Turn 2: Follow-up that references the previous answer
# Note: "it" refers to the Transformer — the system must use history to resolve this
q2 = "What problem was it designed to solve?"
print(f"User: {q2}")
a2 = chat_with_rag(q2)
print(f"Assistant: {a2}")

User: What problem was it designed to solve?


Assistant: The Transformer architecture was designed to solve performance issues associated with older recurrent neural network (RNN) designs for natural language processing (NLP), particularly the bottleneck problem of fixed-size output vectors and difficulties in processing long-distance dependencies in sequences.


In [30]:
# Turn 3: Another follow-up
q3 = "How does the self-attention mechanism achieve this?"
print(f"User: {q3}")
a3 = chat_with_rag(q3)
print(f"Assistant: {a3}")

User: How does the self-attention mechanism achieve this?


Assistant: The self-attention mechanism allows each element in the input sequence to attend to all other elements, enabling the model to capture global dependencies. This capability helps the model understand the context and relationships within the data, facilitating the processing of entire sequences of text at once.


In [31]:
# Print the full conversation history
print("=== Full Conversation History ===")
for msg in chat_history:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    print(f"\n[{role}]: {msg.content[:300]}")

=== Full Conversation History ===

[User]: What is the Transformer architecture?

[Assistant]: The Transformer architecture is a deep learning model developed for natural language processing (NLP). It was introduced in the paper "Attention Is All You Need" in 2017 and solved many performance issues associated with older recurrent neural network (RNN) designs. The architecture uses an attentio

[User]: What problem was it designed to solve?

[Assistant]: The Transformer architecture was designed to solve performance issues associated with older recurrent neural network (RNN) designs for natural language processing (NLP), particularly the bottleneck problem of fixed-size output vectors and difficulties in processing long-distance dependencies in sequ

[User]: How does the self-attention mechanism achieve this?

[Assistant]: The self-attention mechanism allows each element in the input sequence to attend to all other elements, enabling the model to capture global dependencies. This capabi

## 8  Inspecting What Gets Retrieved

Understanding *why* a RAG system gives a particular answer requires looking at the retrieved chunks. A retrieval-augmented answer is only as good as the chunks it receives — garbage in, garbage out.

Good diagnostics:
- Are the retrieved chunks from the correct source?
- Are they relevant to the question?
- Is there redundancy (same passage retrieved twice)?

In [32]:
def rag_with_sources(question: str):
    """Run RAG and print both the answer and the source chunks used."""
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question,
    })
    
    print(f"Question: {question}")
    print("-" * 60)
    print("ANSWER:")
    print(answer)
    print()
    print("RETRIEVED CONTEXT:")
    for i, doc in enumerate(retrieved):
        title = doc.metadata.get("title", "unknown")
        print(f"  [{i+1}] {title}: {doc.page_content[:150]}...")
    print("=" * 60)

rag_with_sources("What is transfer learning and how is it used in NLP?")

Question: What is transfer learning and how is it used in NLP?
------------------------------------------------------------
ANSWER:
Transfer learning (TL) is a technique in machine learning (ML) where knowledge learned from one task is reused to improve performance on a related task. For example, knowledge gained from recognizing cars can help in recognizing trucks. In the context of natural language processing (NLP), transfer learning can be applied through various methods such as Document AI, which allows users without prior AI or ML experience to train systems to extract specific data from documents. This enables non-technical teams to access information hidden in documents efficiently.

[Sources: Transfer learning, Natural language processing]

RETRIEVED CONTEXT:
  [1] Transfer learning: Transfer learning (TL) is a technique in machine learning (ML) in which knowledge learned from a task is re-used in order to boost performance on a re...
  [2] Transfer learning: In 2020, it was di

In [33]:
# Test with a question outside our knowledge base — the model should admit ignorance
rag_with_sources("What is the population of Buenos Aires?")

Question: What is the population of Buenos Aires?
------------------------------------------------------------
ANSWER:
I don't have enough information to answer that.

RETRIEVED CONTEXT:
  [1] Lipizzan: The Lipizzan breed suffered a setback to its population when a viral epidemic hit the Piber Stud in 1983. Forty horses and 8% of the expected foal cro...
  [2] Lipizzan: Until the eighteenth century, Lipizzans had other coat colors, including dun, bay, chestnut, black, piebald, and skewbald. However, gray is a dominant...
  [3] Natural language processing: Lexical semantics
What is the computational meaning of individual words in context?
Distributional semantics
How can we learn semantic representations...
  [4] Transformer (deep learning): In general, there are three classes of language modelling tasks: "masked", "autoregressive", and "prefixLM". These classes are independent of a specif...
  [5] Natural language processing: Document AI
A Document AI platform sits on top of the NLP te

## 9  Chunking Strategy Comparison

The choice of text splitter affects retrieval quality. Let's compare:

| Splitter | How it splits | Best for |
|---|---|---|
| `RecursiveCharacterTextSplitter` | Tries paragraph → sentence → char boundaries | General prose |
| `TokenTextSplitter` | Splits by token count (using a tokenizer) | Fitting the LLM context window precisely |

The `TokenTextSplitter` is more precise when you care about a strict token budget (e.g. the LLM's context window), since character length and token count don't map 1:1 (a character with an accent may tokenise to 2 tokens). We compare both using the same BM25 retriever, so any difference comes purely from the chunk boundaries.

In [34]:
# Token-based splitting
token_splitter = TokenTextSplitter(
    chunk_size=200,     # 200 tokens per chunk (~150 words)
    chunk_overlap=20,
)
token_chunks = token_splitter.split_documents(all_docs)

print(f"RecursiveCharacter (800 chars) : {len(chunks)} chunks")
print(f"Token (200 tokens)             : {len(token_chunks)} chunks")

# Show first chunk from each
print("\n--- RecursiveCharacter chunk ---")
print(chunks[0].page_content[:300])
print("\n--- Token chunk ---")
print(token_chunks[0].page_content[:300])

RecursiveCharacter (800 chars) : 380 chunks
Token (200 tokens)             : 382 chunks

--- RecursiveCharacter chunk ---
Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved t

--- Token chunk ---
Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved t


In [35]:
# Build a second BM25 retriever over the token-based chunks and compare retrieval.
# Same retriever + tokeniser, different chunking — any difference is down to chunk boundaries.
token_retriever = BM25Retriever.from_documents(token_chunks, preprocess_func=bm25_preprocess)
token_retriever.k = 4

test_query = "How does BERT pre-training work?"

char_results = retriever.invoke(test_query)          # BM25 over 800-char chunks
tok_results  = token_retriever.invoke(test_query)    # BM25 over 200-token chunks

print("=== Character-based chunks (BM25) ===")
for d in char_results:
    print(f"  {d.metadata.get('title','?')} | {len(d.page_content)} chars")

print("\n=== Token-based chunks (BM25) ===")
for d in tok_results:
    print(f"  {d.metadata.get('title','?')} | {len(d.page_content)} chars")

=== Character-based chunks (BM25) ===
  Generative pre-trained transformer | 737 chars
  BERT (language model) | 796 chars
  BERT (language model) | 213 chars
  BERT (language model) | 507 chars
  Transfer learning | 294 chars

=== Token-based chunks (BM25) ===
  BERT (language model) | 772 chars
  BERT (language model) | 957 chars
  BERT (language model) | 774 chars
  Generative pre-trained transformer | 1174 chars


## 10  Summary

You have built a complete RAG system. Here is what each component does:

```
+-----------------------------------------------------------------+
|                         INDEXING PHASE                           |
|                                                                 |
|   Documents   ->   Text Splitter   ->   BM25 index              |
|  (Wikipedia)      (chunk_size=800)     (lexical, local)         |
+-----------------------------------------------------------------+

+-----------------------------------------------------------------+
|                        QUERY PHASE (LCEL)                        |
|                                                                 |
|  Question --> BM25 Retriever (k=5)                             |
|       |              |                                          |
|       |        Top-k Chunks                                     |
|       |              |                                          |
|       +------> Prompt Template --> LLM --> StrOutputParser      |
|                {context, question}   (gpt-5.1 via API)      |
+-----------------------------------------------------------------+
```

**Key design decisions and their effects:**

| Decision | Options                                  | Impact |
|---|------------------------------------------|---|
| Chunk size | 200-2000 chars                           | Smaller = more precise; larger = more context per chunk |
| Overlap | 0-20% of chunk size                      | Reduces boundary artefacts |
| Splitter | character vs. token                      | Token splitting respects a strict token budget |
| k (num chunks) | 3-10                                     | More context = better answers, but higher latency and risk of dilution |
| LLM | `gpt-5.1` / `gpt-oss-120b` / other model | Answer quality vs. cost |

**Note on the retriever.** We used **BM25**, a purely lexical retriever that needs no embedding model or external service. Its limitation is that it matches *words*, not *meaning*: a query that shares no vocabulary with the relevant passage (pure paraphrase) can be missed. In production, teams often add a **dense** (embedding-based) retriever and combine the two — a *hybrid* retriever — to get both keyword precision and semantic recall.